# HELIX & HQRM: Diagnostic Math

**HELIX** (Helical Extraction & Locked-In Isotropy eXclusion) phase-locks to the rotating magnetic island O-point, unwraps Boozer coordinates, and produces a focal singularity heat map.

**HQRM** (Helical Quadtree Rotational Mapper) recursively subdivides squares aligned to field-line pitch and locks a 7×7 kernel when local variance drops below 7%.

## Lock-in demodulation

For reference phase $\phi_0$ and poloidal angles $\theta_i$:

$$I = \frac{1}{N}\sum_i s_i \cos(\theta_i - \phi_0), \quad Q = \frac{1}{N}\sum_i s_i \sin(\theta_i - \phi_0)$$

$$A = \sqrt{I^2 + Q^2}$$

Phase-synchronous averaging over $N_{cycles}$ rotations subtracts incoherent turbulence.

---

## Extended Kalman O-point tracker (4-state EKF)

State vector $\mathbf{x} = [\theta,\, \omega,\, A,\, \phi]^\top$ tracks poloidal phase, rotation rate, island amplitude, and toroidal phase.

**Prediction** (discrete, step $\Delta t$):
$$\mathbf{x}_{k|k-1} = F(\Delta t)\, \mathbf{x}_{k-1}, \quad F = \begin{pmatrix} 1 & \Delta t & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & \Delta t/2 & 0 & 1 \end{pmatrix}$$

**Measurement** $z = [\theta_{meas}, \phi_{meas}, A_{meas}]^\top$ with $H$ selecting $(\theta, A, \phi)$.

**Update** (standard EKF): $K = P H^\top (H P H^\top + R)^{-1}$, $\mathbf{x} \leftarrow \mathbf{x} + K(z - H\mathbf{x})$.

Phase wrapping on $\theta, \phi$ enforces toroidal periodicity — this is the **abstract filtering** layer that makes lock-in demodulation optimal for a single rotating coherent mode.

## Boozer unwrap & field-line pitch

Given minor radius $r$ and safety factor $q(r)$:
$$\phi_{Boozer} = \frac{\theta}{q(r)}, \qquad \text{pitch}(r) = \arctan\!\left(\frac{B_\theta}{B_\phi}\right) \approx \arctan\!\left(\frac{1}{q(r)}\right)$$

HQRM rotates each quadtree square by this pitch — **differential geometry on a tokamak flux surface**, not a Cartesian CNN.

## HQRM convergence criterion (7×7 variance lock)

Recursive split while magnetic shear $|dq/dr| > \tau_{shear}$. At depth 7, collect 49 leaf squares $\{S_i\}$. Define heat samples $h_i$. Lock when:
$$\mathrm{Var}(h_1,\ldots,h_{49}) < 0.07$$

**Interpretation:** Monte-Carlo dropout across 49 field-aligned patches — variance below 7% certifies a **crystalline O-point lock** (VISION novel contribution in computational geometry + signal processing).


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if not (repo / "src" / "deepiri_fuselk").exists() and (repo.parent / "src" / "deepiri_fuselk").exists():
    repo = repo.parent

try:
    import deepiri_fuselk  # noqa: F401
except ImportError:
    sys.path.insert(0, str(repo / "src"))
    import deepiri_fuselk  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
%config InlineBackend.figure_formats = ["retina"]


In [ ]:
from deepiri_fuselk.focal.lockin_amplifier import lockin_demodulate, subtract_incoherent_noise
from deepiri_fuselk.helix import HelixEngine, run_hqrm
from deepiri_fuselk.sim import generate_ece_shot

shot = generate_ece_shot(size=48, seed=7, island_amplitude=0.7)
engine = HelixEngine(rotation_hz=5000.0)
result = engine.process(shot.heat_field, shot.raw_signal, shot.angles)

amp, i_comp, q_comp = lockin_demodulate(
    shot.raw_signal, result.o_point[0], shot.angles
)

print(f"Lock-in amplitude: {amp:.3f}  (I={i_comp:.3f}, Q={q_comp:.3f})")
print(f"Phase-locked SNR: {result.phase_locked_snr:.2f}")
print(f"HQRM converged: {result.hqrm.converged}, variance: {result.hqrm.heat_variance:.4f}")
print(f"O-point: {result.o_point}, fracture vector: {result.fracture_vector}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im0 = axes[0].imshow(shot.heat_field, origin="lower", cmap="inferno")
axes[0].set_title("Raw ECE heat field")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(result.focal_map, origin="lower", cmap="magma")
axes[1].set_title("HELIX focal map")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

axes[2].plot(shot.angles, shot.raw_signal, alpha=0.5, label="raw")
cleaned = subtract_incoherent_noise(shot.raw_signal, shot.angles, engine.rotation_hz)
axes[2].plot(shot.angles, cleaned, label="phase-locked avg")
axes[2].set_xlabel("poloidal angle"); axes[2].legend()
axes[2].set_title("Lock-in noise subtraction")

plt.tight_layout()
plt.show()


## Experiment: SNR vs. island amplitude

In [ ]:
amplitudes = np.linspace(0.1, 1.0, 12)
snrs = []

for amp in amplitudes:
    s = generate_ece_shot(32, seed=42, island_amplitude=amp)
    r = HelixEngine().process(s.heat_field, s.raw_signal, s.angles)
    snrs.append(r.phase_locked_snr)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(amplitudes, snrs, "s-", color="C4")
ax.set_xlabel("Island amplitude"); ax.set_ylabel("Phase-locked SNR")
ax.set_title("HELIX denoising gain vs. precursor strength")
plt.tight_layout()
plt.show()


## HQRM quadtree on synthetic field

In [ ]:
n = 64
grid = np.linspace(-1, 1, n)
X, Y = np.meshgrid(grid, grid)
heat = np.exp(-((X - 0.2) ** 2 + (Y + 0.1) ** 2) / 0.08)
hqrm = run_hqrm(heat, variance_threshold=0.07)

print(f"HQRM kernel squares: {len(hqrm.kernel)}, converged: {hqrm.converged}")
print(f"Heat variance: {hqrm.heat_variance:.4f}, O-point: {hqrm.o_point}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(heat, origin="lower", extent=[-1, 1, -1, 1], cmap="inferno", alpha=0.8)
for sq in hqrm.kernel:
    ax.plot(
        [sq.x_min, sq.x_max, sq.x_max, sq.x_min, sq.x_min],
        [sq.y_min, sq.y_min, sq.y_max, sq.y_max, sq.y_min],
        "w-", lw=0.6, alpha=0.7,
    )
ax.plot(*hqrm.o_point, "c*", ms=12, label="O-point")
ax.set_title("HQRM 7×7 kernel lock")
ax.legend()
plt.tight_layout()
plt.show()


## Kalman filter matrices (implemented in `kalman_tracker.py`)

In [ ]:
from deepiri_fuselk.helix.kalman_tracker import PhaseLockedTracker

tracker = PhaseLockedTracker(omega_init=5000.0)
dt = 1e-4
F = tracker._F(dt)
H = np.array([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])

print("State transition F(Δt):")
print(F)
print("\nMeasurement matrix H:")
print(H)
print(f"\nProcess noise diag Q: {np.diag(tracker.Q)}")
print(f"Measurement noise diag R: {np.diag(tracker.R)}")


## Spiral attention — helical RoPE on field-line pitch

In [ ]:
from deepiri_fuselk.focal.spiral_attention import apply_spiral_attention, spiral_attention_weights
from deepiri_fuselk.helix.coordinate_mapper import boozer_map, field_line_pitch, q_profile

r = np.linspace(0.1, 1.0, 64)
pitch = field_line_pitch(r, np.zeros_like(r))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(r, q_profile(r), label="q(r)")
axes[0].plot(r, pitch, label="field-line pitch")
axes[0].legend(); axes[0].set_title("Safety factor → HQRM rotation angle")

feat = result.focal_map[:32, :32]
attended = apply_spiral_attention(feat, pitch=0.6)
axes[1].imshow(attended, origin="lower", cmap="magma")
axes[1].set_title("Spiral attention output (helical mixing)")
plt.tight_layout()
plt.show()
